In [1]:
!pip install crewai  crewai_tools   yagmail

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 5.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 701.4/701.4 kB 17.0

In [2]:
import os
from google.colab import userdata
os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"
os.environ["OPENAI_API_KEY"]=userdata.get('OPENAI_API_KEY')
os.environ["SERPER_API_KEY"]=userdata.get('SERPER_API_KEY')

In [3]:
gmailpass=userdata.get('GMAIL_APP_PASSWORD')


In [4]:
from crewai import Agent, Task, Crew, LLM
from crewai_tools import SerperDevTool

In [5]:
serper_tool = SerperDevTool(n_results=5)

In [6]:
research_agent = Agent(
    name="research-agent",
    role="""
    Gathers facts, key points, and relevant web information for a given email topic.
    """,
    backstory="Collects structured email-related research efficiently.",
    goal="Provide clear, accurate, and structured context for email writing.",
    #llm=llm,
    tools=[serper_tool],
    # verbose=True,
    allow_delegation=True
)

In [7]:

writer_agent = Agent(
    name="writer-agent",
    role="""
    Takes the context provided by the Research Agent (facts, key points, insights)
    and writes a highly professional, clear, and concise email.
    """,
    backstory="Transforms research output into polished, ready-to-send emails.",
    goal="Produce professional emails using the research context provided.",
    #llm=llm,
    # verbose=True,
    allow_delegation=True
)

In [8]:
def output(topic:str):
  research_task = Task(
    name="research-email-topic",
    description=f"Collect facts, key points, and web-based context for the email topic:{topic}",
    expected_output="Bullet points, key insights, and structured context",
    agent=research_agent
)

  write_email_task = Task(
    name="write-email",
    description="""Write a professional email using the context provided by Research Agent.
    """,
    expected_output=

    """A highly professional, polished, concise, and ready-to-send email:
    - Formal tone suitable for business communication
    mention the subject
    - Well-structured with greeting, body, and closing
    - Highlights key points from research context
    - Clear, courteous, and correct
    word limit 50-100
    """,
    context=[research_task],
    agent=writer_agent

)
  crew=Crew(
      agents=[research_agent,writer_agent],
      tasks=[research_task,write_email_task]
  )

  return crew.kickoff()

In [9]:
import json
def generate_email(topic):
  a=output(topic)
  v=str(a)
  subject=v[v.index(":")+1:v.index("\n")]
  body=v[v.index("\n"):]
  return subject,body



In [10]:
import yagmail

In [11]:
def send_email(receiver, subject, content):
    yag = yagmail.SMTP("anushreeiimlucknow@gmail.com",gmailpass)
    yag.send(
        to=receiver,
        subject=subject,
        contents=content
    )

    return f"Email sent to {receiver}"

In [12]:
import gradio as gr

In [13]:
with gr.Blocks() as demo:
  inputbox=gr.Textbox(label="prompt here to generate the email",lines=2)
  btn=gr.Button("Generate")
  subject=gr.Textbox(label="Generated subject",interactive=True,lines=1)
  body=gr.Textbox(label="Generated email",interactive=True,lines=10)
  address=gr.Textbox(label="enter the gmail of the reciever")
  send=gr.Button("send")
  btn.click(generate_email, inputs=inputbox,outputs=[subject,body])
  send.click(send_email,inputs=[address,subject,body], outputs=address )
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f5f60bdeea0c4305cb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f5f60bdeea0c4305cb.gradio.live
